# 05 — Attach metadata and find useful associations

Attach subject-level DIABIMMUNE metadata to the TEMPTED subject scores.

The continuous metadata variables are selected explicitly. Associations use Spearman correlation and are used to identify informative plots in notebook 06.


In [1]:
from datetime import datetime
from pathlib import Path

import pandas as pd
from scipy.stats import spearmanr

root = Path(".") if Path("data").exists() else Path("..")
prep = sorted((root / "data" / "preprocessing").iterdir())[-1]
t_dir = sorted((root / "data" / "tempted").iterdir())[-1]
output = root / "data" / "metadata_analysis" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

# load TEMPTED outputs and metadata
metadata = pd.read_csv(
    prep / "metadata.csv",
    dtype={"sample_id": str, "subject_id": str},
)
tempted = pd.read_csv(
    t_dir / "subject_scores.csv",
    dtype={"subject_id": str},
)

t_dims = [c for c in tempted if c.startswith("component_")]

# continuous subject-level metadata
subject_variables = [
    "gest_time",
    "bf_length",
    "num_abx_treatments",
    "num_abx_first_year",
    "num_aabs",
    "totalige_log",
]


In [2]:
# attach one metadata row to each TEMPTED subject

subject_metadata = (
    metadata[
        ["subject_id", "country"] + subject_variables
    ]
    .drop_duplicates("subject_id")
)

tempted = tempted.merge(
    subject_metadata,
    on="subject_id",
    how="left",
)

tempted.to_csv(
    output / "tempted_with_metadata.csv",
    index=False,
)


In [3]:
# calculate TEMPTED metadata associations

rows = []

for variable in subject_variables:
    values = pd.to_numeric(
        tempted[variable],
        errors="coerce",
    )

    for dim in t_dims:
        keep = values.notna() & tempted[dim].notna()

        rho = spearmanr(
            values[keep],
            tempted.loc[keep, dim],
        ).statistic

        rows.append([
            "TEMPTED",
            variable,
            dim,
            rho,
            abs(rho),
        ])

associations = pd.DataFrame(
    rows,
    columns=[
        "method",
        "metadata",
        "dimension",
        "association",
        "absolute_association",
    ],
)

associations.to_csv(
    output / "metadata_associations.csv",
    index=False,
)

print("Saved:", output)

associations.sort_values(
    "absolute_association",
    ascending=False,
)


Saved: ../data/metadata_analysis/20260816_182056


,method,metadata,dimension,association,absolute_association
11,TEMPTED,totalige_log,component_2,0.266508,0.266508
5,TEMPTED,num_abx_treatments,component_2,0.232320,0.232320
7,TEMPTED,num_abx_first_year,component_2,0.162924,0.162924
4,TEMPTED,num_abx_treatments,component_1,-0.118876,0.118876
2,TEMPTED,bf_length,component_1,0.109044,0.109044
3,TEMPTED,bf_length,component_2,-0.079928,0.079928
0,TEMPTED,gest_time,component_1,-0.077552,0.077552
1,TEMPTED,gest_time,component_2,-0.046893,0.046893
6,TEMPTED,num_abx_first_year,component_1,0.032447,0.032447
10,TEMPTED,totalige_log,component_1,0.024057,0.024057
